# V1 Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path('../Data')
train = pd.read_csv(DATA_DIR / 'train_2025.csv')
test = pd.read_csv(DATA_DIR / 'test_2025.csv')

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Fraud rate: {train['fraud'].mean():.2%}")

Train: (18000, 25)  |  Test: (12000, 24)
Fraud rate: 15.82%


In [2]:
# ======================
# A. 운전자 리스크 피처
# ======================

def add_driver_risk_features(df: pd.DataFrame,
                              age_mean: float = None,
                              age_std: float = None,
                              vehicle_income_p75: float = None) -> pd.DataFrame:
    """
    A 카테고리: 운전자 리스크 피처 생성

    fit 파라미터는 train에서 계산 후 test에도 동일하게 적용 (leakage 방지)

    생성 피처:
      A1. age_outlier          - 나이 이상치 플래그
      A2. age_extreme          - 나이 극단값 플래그
      A3. safety_inv           - 안전등급 역수 (낮을수록 위험)
      A4. safety_risk_flag     - 안전등급 50 미만 플래그
      A5. vehicle_income_ratio - 소득 대비 차량가격 비율
      A6. low_income_high_car  - 저소득 + 고가 차량 불일치 플래그
    """
    df = df.copy()

    # 나이 cap 처리
    df["age_of_driver"] = df["age_of_driver"].clip(upper=100)

    # A1. 나이 이상치 플래그
    # cap 처리 후 평균+2표준편차 초과 나이를 이상치로 표시
    if age_mean is None:
        age_mean = df["age_of_driver"].mean()
    if age_std is None:
        age_std = df["age_of_driver"].std()

    age_upper = age_mean + 2 * age_std
    df["age_outlier"] = (df["age_of_driver"] > age_upper).astype(int)

    # A2. 나이 극단값 플래그
    # 80세 초과
    df["age_extreme"] = (df["age_of_driver"] > 80).astype(int)

    # A3. 안전등급 역수
    # safty_rating이 낮을수록 위험한 운전자
    df["safety_inv"] = 1 / (df["safty_rating"] + 1)

    # A4. 안전등급 위험 플래그
    df["safety_risk_flag"] = (df["safty_rating"] < 50).astype(int)

    # A5. 소득 대비 차량가격 비율
    # 연소득 대비 차량 가격이 높을수록 소득 불일치 의심
    df["vehicle_income_ratio"] = df["vehicle_price"] / df["annual_income"]

    # A6. 저소득 + 고가 차량 불일치 플래그
    # 소득 하위 25% 인데 차량가격 상위 25%
    if vehicle_income_p75 is None:
        vehicle_income_p75 = df["vehicle_income_ratio"].quantile(0.75)

    df["low_income_high_car"] = (
        df["vehicle_income_ratio"] > vehicle_income_p75
    ).astype(int)

    return df, age_mean, age_std, vehicle_income_p75

In [3]:
# ==================
# B. 청구 행동 피처
# ==================

def add_claim_behaviour_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    B 카테고리: 청구 행동 피처 생성

    생성 피처:
      B1. high_claim_freq       - 과거 청구 5회 이상 플래그
      B2. claim_freq_score      - 청구 횟수 구간 점수 (0~3점)
      B3. no_witness            - 목격자 부재 플래그
      B4. no_witness_repeat     - 목격자 없음 + 다중 청구 복합
      B5. mid_liab              - 중간 책임 비율 구간 플래그
      B6. witness_missing       - 목격자 정보 결측 플래그
    """
    df = df.copy()

    # B1. 과거 청구 5회 이상 플래그
    df["high_claim_freq"] = (df["past_num_of_claims"] >= 5).astype(int)

    # B2. 청구 횟수 구간 점수
    # 0점: 0~2회 (정상), 1점: 3~4회, 2점: 5~7회, 3점: 8회 이상
    df["claim_freq_score"] = pd.cut(
        df["past_num_of_claims"],
        bins=[-1, 2, 4, 7, 999],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # B3. 목격자 부재 플래그
    # 결측값(NaN)은 0으로 처리 (정보 없음 = 목격자 없음으로 보수적 처리)
    df["no_witness"] = (
        df["witness_present_ind"].fillna(0) == 0
    ).astype(int)

    # B4. 목격자 없음 + 다중 청구
    df["no_witness_repeat"] = (
        (df["witness_present_ind"].fillna(0) == 0) &
        (df["past_num_of_claims"] >= 5)
    ).astype(int)

    # B5. 중간 책임 비율 구간 플래그
    df["mid_liab"] = (
        (df["liab_prct"] > 18) &
        (df["liab_prct"] <= 82)
    ).astype(int)

    # B6. 목격자 정보 결측 플래그
    df["witness_missing"] = df["witness_present_ind"].isna().astype(int)

    return df

In [4]:
# ===============
# 통합 실행 함수
# ===============

def run_feature_engineering(train: pd.DataFrame,
                             test: pd.DataFrame):

    # A 피처: train fit
    train, age_mean, age_std, vir_p75 = add_driver_risk_features(train)

    # A 피처: test transform (train 파라미터 그대로 사용)
    test, _, _, _ = add_driver_risk_features(
        test,
        age_mean=age_mean,
        age_std=age_std,
        vehicle_income_p75=vir_p75
    )

    # B 피처: train & test 동일 적용
    train = add_claim_behaviour_features(train)
    test  = add_claim_behaviour_features(test)

    return train, test

In [5]:
train, test = run_feature_engineering(train, test)

In [6]:
# ==========
# 내부 유틸
# ==========
def _safe_ratio(num, den, eps=1e-5):
    return num / (den + eps)


def _map_empirical_percentile(train_values: pd.Series, values: pd.Series) -> pd.Series:
    """
    train 분포 기준으로 values의 empirical percentile(0~1)을 매핑
    """
    train_arr = pd.Series(train_values).dropna().to_numpy()
    if len(train_arr) == 0:
        return pd.Series(np.nan, index=values.index)

    train_arr = np.sort(train_arr)
    pos = np.searchsorted(train_arr, values.to_numpy(), side="right")
    pct = pos / len(train_arr)
    return pd.Series(pct, index=values.index)

In [7]:
# ================================================
# C + D 피처 fit / transform
# train에서 파라미터를 구하고, test에는 동일 적용
# ================================================
def fit_cd_feature_params(train_df: pd.DataFrame) -> dict:
    """
    train 데이터에서만 통계량/threshold를 계산
    """
    tmp = train_df.copy()

    # C-1. price_per_weight 관련
    tmp["price_per_weight"] = _safe_ratio(tmp["vehicle_price"], tmp["vehicle_weight"])
    pw_stats = tmp.groupby("vehicle_category")["price_per_weight"].agg(["mean", "std"])
    pw_stats.columns = ["pw_cat_mean", "pw_cat_std"]

    tmp = tmp.merge(pw_stats, on="vehicle_category", how="left")
    tmp["price_weight_zscore"] = (
        (tmp["price_per_weight"] - tmp["pw_cat_mean"]) / (tmp["pw_cat_std"] + 1e-5)
    )

    # C-2. vehicle_price z-score 관련
    vp_stats = tmp.groupby("vehicle_category")["vehicle_price"].agg(["mean", "std"])
    vp_stats.columns = ["price_cat_mean", "price_cat_std"]

    tmp = tmp.merge(vp_stats, on="vehicle_category", how="left")
    tmp["vehicle_price_zscore"] = (
        (tmp["vehicle_price"] - tmp["price_cat_mean"]) / (tmp["price_cat_std"] + 1e-5)
    )

    # C-3. payout_age z-score 관련
    age_stats = tmp.groupby("age_of_vehicle")["claim_est_payout"].agg(["mean", "std"])
    age_stats.columns = ["payout_age_mean", "payout_age_std"]

    tmp = tmp.merge(age_stats, on="age_of_vehicle", how="left")
    tmp["payout_per_vehicle_age"] = _safe_ratio(tmp["claim_est_payout"], tmp["age_of_vehicle"] + 1)
    tmp["payout_age_zscore"] = (
        (tmp["claim_est_payout"] - tmp["payout_age_mean"]) / (tmp["payout_age_std"] + 1e-5)
    )

    # C-4. ratio 관련
    tmp["payout_to_vehicle_price_ratio"] = _safe_ratio(tmp["claim_est_payout"], tmp["vehicle_price"])

    # C 계열 threshold
    threshold_pw = tmp["price_weight_zscore"].quantile(0.95)
    threshold_vp = tmp["vehicle_price_zscore"].quantile(0.95)
    threshold_pa = tmp["payout_age_zscore"].quantile(0.95)
    threshold_pvp = tmp["payout_to_vehicle_price_ratio"].quantile(0.95)

    tmp["price_weight_anomaly_flag"] = (tmp["price_weight_zscore"] > threshold_pw).astype(int)
    tmp["vehicle_price_anomaly_flag"] = (tmp["vehicle_price_zscore"] > threshold_vp).astype(int)
    tmp["payout_age_anomaly_flag"] = (tmp["payout_age_zscore"] > threshold_pa).astype(int)
    tmp["payout_price_ratio_flag"] = (tmp["payout_to_vehicle_price_ratio"] > threshold_pvp).astype(int)

    tmp["vehicle_anomaly_score"] = (
        tmp["price_weight_anomaly_flag"]
        + tmp["vehicle_price_anomaly_flag"]
        + tmp["payout_age_anomaly_flag"]
        + tmp["payout_price_ratio_flag"]
    )

    # D-1 ~ D-4 score 생성
    tmp["safety_risk"] = (100 - tmp["safty_rating"]) / 100
    tmp["log_past_claims"] = np.log1p(tmp["past_num_of_claims"])
    tmp["driver_risk_score"] = tmp["log_past_claims"] * tmp["safety_risk"]

    tmp["vehicle_price_quantile"] = _map_empirical_percentile(tmp["vehicle_price"], tmp["vehicle_price"])
    tmp["vehicle_risk_score"] = (
        tmp["vehicle_price_quantile"] * (1 + tmp["vehicle_anomaly_score"])
    )

    tmp["log_claim_payout"] = np.log1p(tmp["claim_est_payout"])
    tmp["liab_ratio"] = tmp["liab_prct"] / 100
    tmp["claim_risk_score"] = (
        tmp["log_claim_payout"] * tmp["liab_ratio"]
        + tmp["payout_age_anomaly_flag"] * 2
        + tmp["payout_price_ratio_flag"] * 2
    )

    tmp["composite_risk_score"] = (
        tmp["driver_risk_score"] * tmp["vehicle_risk_score"] * tmp["claim_risk_score"]
    )
    tmp["log_composite_risk"] = np.log1p(tmp["composite_risk_score"])

    # D 계열 threshold
    driver_risk_q75 = tmp["driver_risk_score"].quantile(0.75)
    vehicle_risk_q75 = tmp["vehicle_risk_score"].quantile(0.75)
    claim_risk_q75 = tmp["claim_risk_score"].quantile(0.75)

    tmp["income_payout_gap"] = _safe_ratio(tmp["annual_income"], tmp["claim_est_payout"] + 1)
    tmp["income_payout_risk"] = 1 / (tmp["income_payout_gap"] + 1e-5)
    income_payout_risk_q95 = tmp["income_payout_risk"].quantile(0.95)

    past_claims_q75 = tmp["past_num_of_claims"].quantile(0.75)

    params = {
        "pw_stats": pw_stats.reset_index(),
        "vp_stats": vp_stats.reset_index(),
        "age_stats": age_stats.reset_index(),
        "pw_global_mean": tmp["price_per_weight"].mean(),
        "pw_global_std": tmp["price_per_weight"].std(),
        "vp_global_mean": tmp["vehicle_price"].mean(),
        "vp_global_std": tmp["vehicle_price"].std(),
        "pa_global_mean": tmp["claim_est_payout"].mean(),
        "pa_global_std": tmp["claim_est_payout"].std(),
        "threshold_pw": threshold_pw,
        "threshold_vp": threshold_vp,
        "threshold_pa": threshold_pa,
        "threshold_pvp": threshold_pvp,
        "driver_risk_q75": driver_risk_q75,
        "vehicle_risk_q75": vehicle_risk_q75,
        "claim_risk_q75": claim_risk_q75,
        "income_payout_risk_q95": income_payout_risk_q95,
        "past_claims_q75": past_claims_q75,
        "vehicle_price_train": train_df["vehicle_price"].copy(),
    }
    return params

In [8]:
def add_cd_features(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    train에서 구한 params를 사용해 C, D 피처 생성
    """
    df = df.copy()

    # C-1. 가격 대비 무게 비율
    df["price_per_weight"] = _safe_ratio(df["vehicle_price"], df["vehicle_weight"])

    pw_stats = params["pw_stats"].copy()
    df = df.merge(pw_stats, on="vehicle_category", how="left")
    df["pw_cat_mean"] = df["pw_cat_mean"].fillna(params["pw_global_mean"])
    df["pw_cat_std"] = df["pw_cat_std"].fillna(params["pw_global_std"])

    df["price_weight_zscore"] = (
        (df["price_per_weight"] - df["pw_cat_mean"]) / (df["pw_cat_std"] + 1e-5)
    )
    df["price_weight_anomaly_flag"] = (
        df["price_weight_zscore"] > params["threshold_pw"]
    ).astype(int)
    df.drop(columns=["pw_cat_mean", "pw_cat_std"], inplace=True)

    # C-2. 카테고리 내 차량 가격 이상치
    vp_stats = params["vp_stats"].copy()
    df = df.merge(vp_stats, on="vehicle_category", how="left")
    df["price_cat_mean"] = df["price_cat_mean"].fillna(params["vp_global_mean"])
    df["price_cat_std"] = df["price_cat_std"].fillna(params["vp_global_std"])

    df["vehicle_price_zscore"] = (
        (df["vehicle_price"] - df["price_cat_mean"]) / (df["price_cat_std"] + 1e-5)
    )
    df["vehicle_price_anomaly_flag"] = (
        df["vehicle_price_zscore"] > params["threshold_vp"]
    ).astype(int)
    df.drop(columns=["price_cat_mean", "price_cat_std"], inplace=True)

    # C-3. 연식 대비 고액 청구
    df["payout_per_vehicle_age"] = _safe_ratio(df["claim_est_payout"], df["age_of_vehicle"] + 1)

    age_stats = params["age_stats"].copy()
    df = df.merge(age_stats, on="age_of_vehicle", how="left")
    df["payout_age_mean"] = df["payout_age_mean"].fillna(params["pa_global_mean"])
    df["payout_age_std"] = df["payout_age_std"].fillna(params["pa_global_std"])

    df["payout_age_zscore"] = (
        (df["claim_est_payout"] - df["payout_age_mean"]) / (df["payout_age_std"] + 1e-5)
    )
    df["payout_age_anomaly_flag"] = (
        df["payout_age_zscore"] > params["threshold_pa"]
    ).astype(int)
    df.drop(columns=["payout_age_mean", "payout_age_std"], inplace=True)

    # C-4. 청구액 / 차량 가격 비율
    df["payout_to_vehicle_price_ratio"] = _safe_ratio(df["claim_est_payout"], df["vehicle_price"])
    df["payout_price_ratio_flag"] = (
        df["payout_to_vehicle_price_ratio"] > params["threshold_pvp"]
    ).astype(int)

    # C-5. 차량 이상 종합 점수
    df["vehicle_anomaly_score"] = (
        df["price_weight_anomaly_flag"]
        + df["vehicle_price_anomaly_flag"]
        + df["payout_age_anomaly_flag"]
        + df["payout_price_ratio_flag"]
    )
    
    # D-1. 운전자 위험도 점수
    df["safety_risk"] = (100 - df["safty_rating"]) / 100
    df["log_past_claims"] = np.log1p(df["past_num_of_claims"])
    df["driver_risk_score"] = df["log_past_claims"] * df["safety_risk"]

    # D-2. 차량 고위험 점수
    df["vehicle_price_quantile"] = _map_empirical_percentile(
        params["vehicle_price_train"],
        df["vehicle_price"]
    )
    df["vehicle_risk_score"] = (
        df["vehicle_price_quantile"] * (1 + df["vehicle_anomaly_score"])
    )

    # D-3. 청구 위험도 점수
    df["log_claim_payout"] = np.log1p(df["claim_est_payout"])
    df["liab_ratio"] = df["liab_prct"] / 100
    df["claim_risk_score"] = (
        df["log_claim_payout"] * df["liab_ratio"]
        + df["payout_age_anomaly_flag"] * 2
        + df["payout_price_ratio_flag"] * 2
    )

    # D-4. 3중 복합 위험 점수
    df["composite_risk_score"] = (
        df["driver_risk_score"] * df["vehicle_risk_score"] * df["claim_risk_score"]
    )
    df["log_composite_risk"] = np.log1p(df["composite_risk_score"])

    # D-5. 고위험 복합 플래그
    driver_high = df["driver_risk_score"] > params["driver_risk_q75"]
    vehicle_high = df["vehicle_risk_score"] > params["vehicle_risk_q75"]
    claim_high = df["claim_risk_score"] > params["claim_risk_q75"]

    df["triple_high_risk_flag"] = (driver_high & vehicle_high & claim_high).astype(int)

    # D-6. 소득 대비 청구액 불일치
    df["income_payout_gap"] = _safe_ratio(df["annual_income"], df["claim_est_payout"] + 1)
    df["income_payout_risk"] = 1 / (df["income_payout_gap"] + 1e-5)

    df["income_payout_anomaly_flag"] = (
        df["income_payout_risk"] > params["income_payout_risk_q95"]
    ).astype(int)

    # D-7. 고위험 운전자 × 이상 차량 교호작용
    df["driver_vehicle_interaction"] = (
        df["log_past_claims"] * df["vehicle_price_zscore"]
    )

    high_claims = df["past_num_of_claims"] > params["past_claims_q75"]
    df["high_claim_driver_anomaly_vehicle"] = (
        high_claims & (df["vehicle_price_anomaly_flag"] == 1)
    ).astype(int)

    return df

In [9]:
def run_feature_engineering(train: pd.DataFrame, test: pd.DataFrame):
    params = fit_cd_feature_params(train)
    train_fe = add_cd_features(train, params)
    test_fe = add_cd_features(test, params)
    return train_fe, test_fe, params

In [10]:
train, test, params = run_feature_engineering(train, test)

In [11]:
# ===============
# 최종 피처 확인
# ===============
new_features_C = [
    "price_per_weight",
    "price_weight_zscore",
    "price_weight_anomaly_flag",
    "vehicle_price_zscore",
    "vehicle_price_anomaly_flag",
    "payout_per_vehicle_age",
    "payout_age_zscore",
    "payout_age_anomaly_flag",
    "payout_to_vehicle_price_ratio",
    "payout_price_ratio_flag",
    "vehicle_anomaly_score",
]

new_features_D = [
    "safety_risk",
    "log_past_claims",
    "driver_risk_score",
    "vehicle_price_quantile",
    "vehicle_risk_score",
    "log_claim_payout",
    "liab_ratio",
    "claim_risk_score",
    "composite_risk_score",
    "log_composite_risk",
    "triple_high_risk_flag",
    "income_payout_gap",
    "income_payout_risk",
    "income_payout_anomaly_flag",
    "driver_vehicle_interaction",
    "high_claim_driver_anomaly_vehicle",
]

print("=" * 60)
print("C. 차량 이상 피처 (11개) - train")
print("=" * 60)
for f in new_features_C:
    non_null = train[f].notna().sum()
    print(f"  {f:<40} | non-null: {non_null}")

print()
print("=" * 60)
print("D. 상호작용 피처 (16개) - train")
print("=" * 60)
for f in new_features_D:
    non_null = train[f].notna().sum()
    print(f"  {f:<40} | non-null: {non_null}")

print()
print(f"총 신규 피처: {len(new_features_C) + len(new_features_D)}개")
print(f"최종 train 데이터 shape: {train.shape}")
print(f"최종 test 데이터 shape: {test.shape}")

C. 차량 이상 피처 (11개) - train
  price_per_weight                         | non-null: 18000
  price_weight_zscore                      | non-null: 18000
  price_weight_anomaly_flag                | non-null: 18000
  vehicle_price_zscore                     | non-null: 18000
  vehicle_price_anomaly_flag               | non-null: 18000
  payout_per_vehicle_age                   | non-null: 18000
  payout_age_zscore                        | non-null: 18000
  payout_age_anomaly_flag                  | non-null: 18000
  payout_to_vehicle_price_ratio            | non-null: 18000
  payout_price_ratio_flag                  | non-null: 18000
  vehicle_anomaly_score                    | non-null: 18000

D. 상호작용 피처 (16개) - train
  safety_risk                              | non-null: 18000
  log_past_claims                          | non-null: 18000
  driver_risk_score                        | non-null: 18000
  vehicle_price_quantile                   | non-null: 18000
  vehicle_risk_score             

In [12]:
# Train / Test 합치기 (타겟 컬럼 제외)
train['_is_train'] = 1
test['_is_train']  = 0

# test에는 fraud 컬럼 없음 → 임시 NaN 추가
if 'fraud' not in test.columns:
    test['fraud'] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined shape: {df.shape}")

Combined shape: (30000, 65)


In [13]:
# 최종 데이터셋 저장
X_train = train.drop(columns=['fraud']).copy()
y_train = train['fraud']
X_test  = test

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}  (fraud={y_train.mean():.2%})")
print(f"X_test:  {X_test.shape}")

# CSV 저장
X_train.assign(fraud=y_train.values).to_csv(DATA_DIR / 'train_V1.csv', index=False)
X_test.to_csv(DATA_DIR / 'test_V1.csv', index=False)

X_train: (18000, 64)
y_train: (18000,)  (fraud=15.82%)
X_test:  (12000, 65)
